# Woche 12: Large Language Modelle lokal ausführen mit Gemma 4 und llama.cpp

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/12_gemma4_llamacpp.ipynb)

In dieser Übung lernst du, wie du das neue Gemma 4 Modell von Google lokal in Google Colab mit Hilfe der Bibliothek `llama.cpp` ausführst. `llama.cpp` ist eine hocheffiziente Implementierung für LLM-Inferenz in C/C++, die durch `llama-cpp-python` bequem in Python genutzt werden kann.

## 1. Setup

Zuerst installieren wir `llama-cpp-python` mit CUDA-Unterstützung, um die GPU in Colab nutzen zu können. Zudem benötigen wir das `huggingface_hub` Paket, um die Modelldateien herunterzuladen.

In [ ]:
# Installation von llama-cpp-python mit GPU-Unterstützung
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python huggingface_hub

## 2. Modell herunterladen

Wir laden das Gemma 4 Modell im GGUF-Format direkt von Hugging Face herunter. Wir nutzen hier die Instruktions-Version mit ca. 2 Milliarden Parametern (`E2B`), die gut in den Speicher von Colab passt.

In [ ]:
from huggingface_hub import hf_hub_download

repo_id = "ggml-org/gemma-4-E2B-it-GGUF"
filename = "gemma-4-E2B-it-Q4_K_M.gguf" # Quantisierte Version (4-bit)

model_path = hf_hub_download(repo_id=repo_id, filename=filename)
print(f"Modell wurde heruntergeladen nach: {model_path}")

## 3. Modell laden

Nun laden wir das Modell mit `llama_cpp.Llama`. Wir setzen `n_gpu_layers=-1`, um alle Schichten auf die GPU auszulagern.

In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1, # Nutze die GPU für alle Schichten
    n_ctx=2048,      # Kontextfenstergröße
)

## 4. Inferenz mit Gemma 4 Prompt-Format

Gemma 4 verwendet ein spezielles Format für Prompts mit Steuerungstokens wie `<|turn|>`, `user` und `model`.

In [ ]:
def generate_response(prompt, system_prompt="Du bist ein hilfreicher Assistent."):
    formatted_prompt = f"<|turn|>system
{system_prompt}<turn|>
<|turn|>user
{prompt}<turn|>
<|turn|>model
"
    
    response = llm(
        prompt=formatted_prompt,
        max_tokens=512,
        stop=["<turn|>"],
        echo=False
    )
    return response["choices"][0]["text"]

user_input = "Erkläre mir kurz, was ein Transformer-Modell ist."
print(generate_response(user_input))

## 5. Thinking-Modus aktivieren

Eine Besonderheit von Gemma 4 ist der "Thinking-Modus". Dieser wird durch das Token `<|think|>` in der Systemanweisung aktiviert. Das Modell generiert dann interne Überlegungen in einem speziellen Thought-Channel, bevor es die finale Antwort gibt.

In [ ]:
system_with_think = "<|think|>Du bist ein hilfreicher Assistent."
complex_question = "Wenn ich 3 Äpfel habe und 2 Birnen esse, wie viele Bananen habe ich dann, wenn ich vorher 5 Bananen gekauft habe?"

print("Antwort mit Thinking-Modus:")
print(generate_response(complex_question, system_prompt=system_with_think))